# hparam-precedence-merge — worked example 1: Three-layer config merge via dict unpacking

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `hparam-precedence-merge`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The idiomatic Python way to merge configuration layers with strict precedence is `{**defaults, **file_cfg, **cli_args}`. Later entries in this dict literal shadow earlier ones: if the same key appears in multiple layers, the rightmost wins. This single expression creates a fresh dict without mutating any of the inputs, making it safe to call multiple times.

## Worked solution

**Step 1 — define three config layers.** `defaults` provides fallback values for every key, `file_cfg` overrides some of them (as if read from a YAML file), and `cli_args` contains whatever the user passed at the command line.

**Step 2 — merge with `{**defaults, **file_cfg, **cli_args}`.** This unpacks all three in order. A key in `file_cfg` shadows the same key in `defaults`; a key in `cli_args` shadows both.

**Step 3 — verify precedence.** Check that `lr` comes from `cli_args`, `batch_size` from `file_cfg`, and `dropout` from `defaults` (since neither `file_cfg` nor `cli_args` set it).

**Step 4 — verify no input mutation.** The three input dicts must be unchanged after the merge.

In [ ]:
import torch as t

t.manual_seed(0)

def merge_config(defaults, file_cfg, cli_args):
    # dict unpacking: rightmost dict wins on key conflicts
    return {**defaults, **file_cfg, **cli_args}

# Three config layers
defaults  = {'lr': 0.01, 'batch_size': 32, 'dropout': 0.1, 'epochs': 100}
file_cfg  = {'lr': 0.001, 'batch_size': 64}             # overrides lr, batch_size
cli_args  = {'lr': 0.005}                               # overrides lr only

result = merge_config(defaults, file_cfg, cli_args)

print(f"lr:         {result['lr']}")         # 0.005 (from cli_args)
print(f"batch_size: {result['batch_size']}")  # 64    (from file_cfg)
print(f"dropout:    {result['dropout']}")    # 0.1   (from defaults)
print(f"epochs:     {result['epochs']}")     # 100   (from defaults)

assert result['lr'] == 0.005
assert result['batch_size'] == 64
assert result['dropout'] == 0.1
assert result['epochs'] == 100

# No mutation
assert defaults == {'lr': 0.01, 'batch_size': 32, 'dropout': 0.1, 'epochs': 100}
assert file_cfg == {'lr': 0.001, 'batch_size': 64}
assert cli_args == {'lr': 0.005}
print("All precedence and non-mutation checks passed.")